<a href="https://colab.research.google.com/github/IvanKatsapov/Algo-MIPT-Homework/blob/main/relevance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

d объектов, n параметров, k самых релевантных объектов.
Хранение O(d*n), + выделение памяти O(k) на запись релевантных
Поиск оптимальных O(klogk)

In [5]:
import heapq

class Ranking:
    def __init__(self, weights, features):
        self.weights = weights
        self.features = features
        self.scores = [sum(a * f for a, f in zip(weights, row)) for row in features] #по формуле
        self.heap = list(range(len(features)))
        self.pos = list(range(len(features))) #сохраняем позицию
        for i in range(len(self.heap) // 2 - 1, -1, -1):
            self._sift_down(i)

    def _better(self, i, j):
      #сравниваем i и j лементы
        a = self.heap[i]
        b = self.heap[j]
        return (self.scores[a], -a) > (self.scores[b], -b)

    def _swap(self, i, j):
      #меняем местами позиции в heap
        self.heap[i], self.heap[j] = self.heap[j], self.heap[i]
        #обновляем позвиции
        self.pos[self.heap[i]] = i
        self.pos[self.heap[j]] = j

    def _sift_up(self, i):
        while i > 0:
            parent = (i - 1) // 2
            if not self._better(i, parent):
                break
            self._swap(i, parent)
            i = parent
            #если оценка объекта увеличилась, проверяем, не стал ли он лучше родителя

    def _sift_down(self, i):
        size = len(self.heap)
        while 2 * i + 1 < size:
            child = 2 * i + 1
            if child + 1 < size and self._better(child + 1, child):
                child += 1
            if not self._better(child, i):
                break
            self._swap(i, child)
            i = child
            #проверяем, если оценка родителя изменилась в меньшу сторону

    def update(self, object_id, feature_id, value):
      #xnj,s vj;yj ,,skj gjvtyznm pyfxtybt d yfitv vfccbdt
        i = object_id - 1
        j = feature_id - 1
        delta = self.weights[j] * (value - self.features[i][j])
        self.features[i][j] = value
        self.scores[i] += delta
        if delta > 0:
            self._sift_up(self.pos[i])
        elif delta < 0:
            self._sift_down(self.pos[i])

    def top(self, k):
        if not 0 <= k <= len(self.heap):
            raise ValueError('k должно быть от 0 до числа объектов')
        if k == 0:
            return []
        # Очередь кандидатов содержит корни ещё не просмотренных поддеревьев.
        # Потомок не может обогнать своего родителя в основной max-куче.
        root = self.heap[0]
        candidates = [(-self.scores[root], root, 0)]
        result = []
        for _ in range(k):
            score, object_id, index = heapq.heappop(candidates)
            result.append(object_id + 1)
            if len(result) == k:
                break
            for child in (2 * index + 1, 2 * index + 2):
                if child < len(self.heap):
                    obj = self.heap[child]
                    heapq.heappush(candidates,
                                   (-self.scores[obj], obj, child))
        return result


def solve_ranking(input_stream=None, output_stream=None):
  n = int(input())
  weights = list(map(int, input().split()))

  d = int(input())
  features = []
  for _ in range(d):
    row = list(map(int, input().split()))
    features.append(row)

  ranking = Ranking(weights, features)

  q = int(input())
  for _ in range(q):
    query = list(map(int, input().split()))
    if query[0] == 1:
      result = ranking.top(query[1])
      print(result)
    else:
      ranking.update(query[1], query[2], query[3])

In [6]:
solve_ranking()

2
1 100
10
1 2
2 1
3 1
4 1
5 1
6 1
7 1
8 1
9 1
10 1
4
1 2
[1, 10]
1 10
[1, 10, 9, 8, 7, 6, 5, 4, 3, 2]
2 4 1 1000
1 10
[4, 1, 10, 9, 8, 7, 6, 5, 3, 2]
